# Phase 1A - TATR table topology (Colab runner)

Runner only: mount, git pull, install, import, call scripts. No logic lives here
(see CLAUDE.md P1/P2). Edit `src/` and `scripts/` locally, push, then re-run.

- **Step 1: inspect** the real FinTabNet.c-Structure layout (already confirmed).
- **Step 2: TATR inference + topology metrics** - run the structure model over a small
  batch, write the prediction/manifest/metrics/failure artifacts, paste the summary back.

## Boot

In [ ]:
# 1. Mount Drive (outputs persist here: manifest / metrics / failures / tables).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Pull the latest code onto the VM (git is the single source of truth).
!cd /content/FinDocStructRAG 2>/dev/null && git pull || git clone https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the environment.
import sys
sys.path.insert(0, '/content/FinDocStructRAG')
from src import config, fintabnet_loader
print('IN_COLAB    :', config.IN_COLAB)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('cache root  :', fintabnet_loader.structure_root())

## Step 1 - inspect the dataset

Downloads + extracts `FinTabNet.c-Structure.tar.gz` to Colab scratch and prints the
layout, file-extension histogram, and a few parsed annotations. Already confirmed; re-run
only if you need to re-check the format.

In [ ]:
# Step 1 needs only huggingface_hub (fast).
!pip install -q huggingface_hub
!python scripts/inspect_fintabnet.py --limit 3

## Step 2 - TATR inference + topology metrics

Installs the GPU stack, then runs the structure model over a small batch. Outputs land
under `outputs/` on Drive (manifest / topology metrics / failures, plus the predicted
tables in `tatr_predicted/`). The run is resumable: re-running skips samples already
marked success in the manifest.

Make sure the runtime is **GPU (T4)**. **Paste back** the final
`processed/skipped/failed` line and the summary JSON (text, not a screenshot).

In [ ]:
# 4. Install the GPU stack for TATR inference (torch/transformers/timm/Pillow/...).
!pip install -q -r requirements-colab.txt

In [ ]:
# 5. Run TATR structure inference + topology metrics over a fixed random subset.
#    --seed gives a reproducible, issuer-unbiased sample (DESIGN_SPEC 18.6):
#    debug=seed 7/limit 50, mvp=seed 42/limit 300, final=seed 2026/limit 500.
#    Seeds are nested, so growing --limit reuses the smaller run (resumable).
!python scripts/run_phase1a_colab.py --limit 300 --seed 42 --run-id mvp_rand

## Step 2b - authoritative metrics (CPU, no GPU)

Recompute the topology report from all persisted predictions in the run's manifest.
Run this after the GPU run; it is safe to re-run and is correct even across resumes
(the GPU runner only scores what one run processed). Set `--run-id` to match.

In [ ]:
# Recompute the topology report from persisted predictions (no GPU needed).
!python scripts/evaluate_tables.py --run-id mvp_rand

## Step 2 results - inspect artifacts

Reads the run's report / manifest / failures. Set `run_id` to match the run cell above.
Failures are shown **newest-first** (the file itself is append-only, oldest-first), so a
truncated output still shows the latest entries - and `failed=0` runs add nothing here.

In [ ]:
# Inspect the latest run's artifacts (display only, no logic).
from src import config

run_id = "mvp_rand"
report   = config.EVALUATION   / f"phase1a_topology_{run_id}.json"
manifest = config.MANIFESTS    / f"phase1a_{run_id}.csv"
failures = config.FAILURE_LOGS / f"phase1a_{run_id}.jsonl"
runlog   = config.MANIFESTS    / "phase1a_runs.jsonl"

print("=== latest run summary ===")
print((runlog.read_text().splitlines() or ["(empty)"])[-1] if runlog.exists() else "(none)")

print("\n=== topology report ===")
print(report.read_text() if report.exists() else "(missing)")

print("\n=== manifest tail (newest 5) ===")
print("\n".join(manifest.read_text().splitlines()[-5:]) if manifest.exists() else "(missing)")

print("\n=== failures (newest 5, newest first) ===")
if failures.exists():
    fl = failures.read_text().splitlines()
    print(f"total={len(fl)}")
    print("\n".join(fl[-5:][::-1]) or "(empty)")
else:
    print("(none)")

## Step 3 - deliverable figures (CPU, no GPU)

Renders the Phase 1A deliverable screenshots (DESIGN_SPEC 5.14) from the persisted artifacts - a clean sample and a failure sample (#8): TATR box overlay, derived cell grid, spanning cells, geometry report, and the predicted-table HTML, plus the metrics summary. #6 (GT-filled HTML) is a Phase 1B deliverable. Set `--run-id` to match the run above.

In [ ]:
# Render figures to outputs/figures/phase1a/ on Drive (prediction only, P4).
!python scripts/render_phase1a_figures.py --run-id mvp_rand

In [ ]:
# Display the rendered figures (display only, no logic).
import importlib
from src import config
importlib.reload(config)  # pick up config changes pulled after the kernel started
from IPython.display import Image as IPyImage, HTML, display

fig_root = config.FIGURES / "phase1a"
for png in sorted(fig_root.rglob("*.png")):
    print(png.relative_to(fig_root)); display(IPyImage(filename=str(png)))
for txt in sorted(fig_root.rglob("05_geometry.txt")):
    print(txt.relative_to(fig_root)); print(txt.read_text())
for h in sorted(fig_root.rglob("*.html")):
    print(h.relative_to(fig_root)); display(HTML(h.read_text()))